In [1]:
from google.colab import drive
drive.mount('/content/drive')

DATASET_PATH = "/content/drive/MyDrive/ravdess"
AUDIO_PATH = f"{DATASET_PATH}/audio"
VIDEO_PATH = f"{DATASET_PATH}/video"
META_PATH = f"{DATASET_PATH}/metadata"


Mounted at /content/drive


In [2]:
import os
import pandas as pd

emotion_map = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

rows = []

for actor in sorted(os.listdir(AUDIO_PATH)):
    actor_dir = os.path.join(AUDIO_PATH, actor)
    if not os.path.isdir(actor_dir):
        continue

    for fname in sorted(os.listdir(actor_dir)):
        if not fname.endswith(".wav"):
            continue

        parts = fname.split("-")
        emotion_code = parts[2]
        emotion = emotion_map[emotion_code]

        audio_rel = os.path.join("audio", actor, fname)
        video_rel = os.path.join("video", actor, fname.replace(".wav", ".mp4"))

        rows.append({
            "actor": actor,
            "fname": fname,
            "audio_path": audio_rel,
            "video_path": video_rel,
            "emotion": emotion
        })

df = pd.DataFrame(rows)
os.makedirs(META_PATH, exist_ok=True)

labels_csv_path = os.path.join(META_PATH, "labels.csv")
df.to_csv(labels_csv_path, index=False)

labels_csv_path


'/content/drive/MyDrive/ravdess/metadata/labels.csv'

In [3]:
!pip install librosa soundfile

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
import pandas as pd
import os

class RavdessAudioDataset(Dataset):
    def __init__(self, csv_path, root_dir, sr=16000, n_mels=64, max_len=400):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.sr = sr
        self.n_mels = n_mels
        self.max_len = max_len

        # Map emotions to integers
        self.label2id = {label: i for i, label in enumerate(sorted(self.df["emotion"].unique()))}
        self.id2label = {v: k for k, v in self.label2id.items()}

    def __len__(self):
        return len(self.df)

    def _fix_length(self, mel_db):
        """
        mel_db: [n_mels, T]
        Make time dimension = self.max_len by padding or truncating.
        """
        n_mels, T = mel_db.shape

        if T < self.max_len:
            # pad at the end with zeros
            pad_width = self.max_len - T
            mel_db = np.pad(mel_db, ((0, 0), (0, pad_width)), mode="constant")
        else:
            # truncate (you can also center-crop if you like)
            mel_db = mel_db[:, :self.max_len]

        return mel_db

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = os.path.join(self.root_dir, row["audio_path"])

        # load audio
        y, sr = librosa.load(audio_path, sr=self.sr)

        # mel spectrogram
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_fft=512,
            hop_length=160,
            n_mels=self.n_mels
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)  # [n_mels, T]

        # normalize
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

        # fix length
        mel_db = self._fix_length(mel_db)  # [n_mels, max_len]

        # to tensor: [1, n_mels, max_len]
        x = torch.tensor(mel_db, dtype=torch.float32).unsqueeze(0)

        y_label = torch.tensor(self.label2id[row["emotion"]], dtype=torch.long)
        return x, y_label


In [5]:
csv_path = labels_csv_path  # from earlier
dataset = RavdessAudioDataset(csv_path, DATASET_PATH)

len(dataset), dataset[0][0].shape, dataset[0][1]

(1440, torch.Size([1, 64, 400]), tensor(5))

In [6]:
class AudioCNN(nn.Module):
    def __init__(self, n_mels=64, n_classes=8):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.fc = nn.Linear(64, n_classes)

    def forward(self, x):
        h = self.conv(x)          # [B, 64, 1, 1]
        h = h.view(h.size(0), -1) # [B, 64]
        out = self.fc(h)
        return out

n_classes = len(dataset.label2id)
model = AudioCNN(n_classes=n_classes).to("cuda" if torch.cuda.is_available() else "cpu")


In [9]:
from torch.utils.data import random_split, DataLoader

dataset_size = len(dataset)
val_size = int(0.2 * dataset_size)
train_size = dataset_size - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)

len(train_dataset), len(val_dataset)

(1152, 288)

In [10]:
from tqdm.auto import tqdm
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 15  # more epochs now

for epoch in range(num_epochs):
    print(f"\n===== Epoch {epoch+1} / {num_epochs} =====")

    # ---- Train ----
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, y in tqdm(train_loader, desc=f"Train {epoch+1}", leave=False):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)
        preds = logits.argmax(dim=1)
        train_correct += (preds == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ---- Validate ----
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X, y in tqdm(val_loader, desc=f"Val {epoch+1}", leave=False):
            X, y = X.to(device), y.to(device)

            logits = model(X)
            loss = criterion(logits, y)

            val_loss += loss.item() * X.size(0)
            preds = logits.argmax(dim=1)
            val_correct += (preds == y).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Train  Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val    Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")


===== Epoch 1 / 15 =====


Train 1:   0%|          | 0/72 [00:00<?, ?it/s]

Val 1:   0%|          | 0/18 [00:00<?, ?it/s]

Train  Loss: 1.9932 | Train Acc: 0.2248
Val    Loss: 1.9673 | Val   Acc: 0.2882

===== Epoch 2 / 15 =====


Train 2:   0%|          | 0/72 [00:00<?, ?it/s]

...

Val 14:   0%|          | 0/18 [00:00<?, ?it/s]

Train  Loss: 1.6532 | Train Acc: 0.3602
Val    Loss: 1.6361 | Val   Acc: 0.3819

===== Epoch 15 / 15 =====


Train 15:   0%|          | 0/72 [00:00<?, ?it/s]

Val 15:   0%|          | 0/18 [00:00<?, ?it/s]

Train  Loss: 1.6445 | Train Acc: 0.3750
Val    Loss: 1.6566 | Val   Acc: 0.3681


In [11]:
import torch

torch.save(model.state_dict(), "ravdess_audio_cnn.pth")
print("Saved model to ravdess_audio_cnn.pth")


Saved model to ravdess_audio_cnn.pth


In [12]:
import numpy as np
import pandas as pd
import torch
import os

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

all_embeddings = []
all_rows = []

with torch.no_grad():
    for i in range(len(dataset)):
        # Get input + label
        x, y = dataset[i]          # x: [1, 64, max_len]
        x = x.unsqueeze(0).to(device)  # [1, 1, 64, max_len]

        # Forward until before final FC
        h = model.conv(x)          # [1, 64, 1, 1]
        h = h.view(h.size(0), -1)  # [1, 64]
        emb = h.squeeze(0).cpu().numpy()  # [64]

        # Metadata from the original df
        row = dataset.df.iloc[i]
        all_rows.append({
            "idx": i,
            "actor": row["actor"],
            "fname": row["fname"],
            "emotion": row["emotion"]
        })
        all_embeddings.append(emb)

# Convert to DataFrame
meta_df = pd.DataFrame(all_rows)
emb_dim = all_embeddings[0].shape[0]
emb_cols = [f"audio_emb_{j}" for j in range(emb_dim)]
emb_df = pd.DataFrame(all_embeddings, columns=emb_cols)

audio_features_df = pd.concat([meta_df, emb_df], axis=1)
audio_features_path = os.path.join(META_PATH, "audio_embeddings.csv")
audio_features_df.to_csv(audio_features_path, index=False)

audio_features_path, audio_features_df.head()

('/content/drive/MyDrive/ravdess/metadata/audio_embeddings.csv',
    idx     actor                     fname  emotion  audio_emb_0  audio_emb_1  \
 0    0  Actor_01  03-01-01-01-01-01-01.wav  neutral     1.529860     0.914648   
 1    1  Actor_01  03-01-01-01-01-02-01.wav  neutral     1.460622     0.949778   
 2    2  Actor_01  03-01-01-01-02-01-01.wav  neutral     1.601629     0.871646   
 3    3  Actor_01  03-01-01-01-02-02-01.wav  neutral     1.835651     0.921633   
 4    4  Actor_01  03-01-02-01-01-01-01.wav     calm     1.113918     1.042680   
 
    audio_emb_2  audio_emb_3  audio_emb_4  audio_emb_5  ...  audio_emb_54  \
 0     0.627405     1.342164          0.0          0.0  ...      0.141293   
 1     0.583477     1.309300          0.0          0.0  ...      0.203736   
 2     0.652081     1.412089          0.0          0.0  ...      0.144319   
 3     0.679228     1.670208          0.0          0.0  ...      0.185169   
 4     0.457832     1.017605          0.0          0.0  

video

In [13]:
!pip install opencv-python

import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
import os

import torchvision
from torchvision import transforms

In [14]:
class RavdessVideoDataset(Dataset):
    def __init__(self, csv_path, root_dir, img_size=224):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir

        # map emotions to ids using same ordering logic as audio
        self.label2id = {label: i for i, label in enumerate(sorted(self.df["emotion"].unique()))}
        self.id2label = {v: k for k, v in self.label2id.items()}

        # image transforms (ImageNet style)
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],  # ImageNet stats
                std=[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.df)

    def _load_middle_frame(self, video_path):
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"Could not open video: {video_path}")

        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        mid_frame_idx = frame_count // 2

        cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame_idx)
        ret, frame = cap.read()
        cap.release()

        if not ret or frame is None:
            raise RuntimeError(f"Could not read frame {mid_frame_idx} from {video_path}")

        # BGR (OpenCV) → RGB
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        return frame

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        rel_video_path = row["video_path"]
        video_path = os.path.join(self.root_dir, rel_video_path)

        frame = self._load_middle_frame(video_path)
        img = self.transform(frame)

        label = self.label2id[row["emotion"]]
        label = torch.tensor(label, dtype=torch.long)

        return img, label

In [15]:
video_dataset = RavdessVideoDataset(labels_csv_path, DATASET_PATH)
len(video_dataset), video_dataset[0][0].shape, video_dataset[0][1]

RuntimeError: Could not open video: /content/drive/MyDrive/ravdess/video/Actor_01/03-01-01-01-01-01-01.mp4